[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tragabytes/panel-viaje/blob/main/tests/fase1_meteo.ipynb)

# Fase 1.3 — Evaluación de Open-Meteo (meteorología)

**Proyecto:** Panel de viaje para coche — https://github.com/tragabytes/panel-viaje  
**Sesión:** 04  
**Fecha:** 7 de abril de 2026

## Objetivo

Validar Open-Meteo como fuente única de meteorología para el panel. Necesitamos confirmar:

1. **Tiempo actual** (temperatura, sensación térmica, humedad, viento, código de tiempo) sobre las 12 coordenadas de la sesión 03 — una por CCAA, para que el banco de pruebas sea coherente con la fase 1.1.
2. **Previsión por horas** para las próximas 12 h en una coordenada, validando que el paso es horario y los campos útiles vienen.
3. **Mapeo completo de `weather_code`** a las 6 categorías visuales que necesita la Vista 2 (soleado, nublado, niebla, lluvia, nieve, tormenta) para las animaciones de fondo.
4. **Latencia y tamaño de respuesta**, comparable con lo medido para Nominatim en la sesión 03.
5. **Coherencia con AEMET**: comparación informal con la temperatura observada en una capital española usando el widget público de AEMET, para descartar desvíos serios.
6. **Decidir qué campos mostramos y cuáles descartamos** para minimizar el payload, en línea con el principio del proyecto de "cada petición cuenta".

Salida del notebook: una recomendación clara (apta / apta con reservas / descartar) que se trasladará a la ficha de API en `seguimiento_desarrollo_panel_viaje.docx`.

## Datos de partida confirmados antes de empezar

- **Endpoint**: `https://api.open-meteo.com/v1/forecast`
- **Autenticación**: ninguna. No requiere User-Agent propio (a diferencia de Nominatim y Photon), pero lo enviamos igual por buena ciudadanía.
- **Formato**: JSON.
- **Modelos para España**: Open-Meteo elige automáticamente entre DWD ICON, ECMWF IFS y otros, sin que tengamos que especificar nada.
- **`weather_code`**: 28 valores documentados, basados en la tabla WMO 4677.
- **Licencia**: CC-BY 4.0, uso no comercial gratuito (encaja con el proyecto).

Estos datos vienen de la documentación oficial (https://open-meteo.com/en/docs) y de la página principal de Open-Meteo, no son inventados.

## 0. Setup

In [ ]:
import requests
import time
import json
from statistics import mean, median

BASE = "https://api.open-meteo.com/v1/forecast"
HEADERS = {
    "User-Agent": "panel-viaje-test/0.1 (https://github.com/tragabytes/panel-viaje)"
}

# 12 coordenadas de la sesión 03 — una por comunidad autónoma + casos especiales.
# Mismo banco de pruebas que en fase1_geocoding.ipynb para mantener coherencia entre fases.
PUNTOS = [
    ("Albarracín (Aragón)",            40.4064,  -1.4433),
    ("Besalú (Cataluña)",              42.1995,   2.6989),
    ("La Alberca (Castilla y León)",   40.4894,  -6.1108),
    ("Chinchón (Madrid)",              40.1376,  -3.4276),
    ("Ronda (Andalucía)",              36.7426,  -5.1654),
    ("Cangas de Onís (Asturias)",      43.3502,  -5.1297),
    ("Santillana del Mar (Cantabria)", 43.3923,  -4.1063),
    ("Hondarribia (País Vasco)",       43.3713,  -1.7979),
    ("Valldemossa (Baleares)",         39.7100,   2.6230),
    ("Garachico (Canarias)",           28.3735, -16.7639),
    ("Olite (Navarra)",                42.4810,  -1.6494),
    ("Laguardia (La Rioja)",           42.5527,  -2.5829),
]

print(f"{len(PUNTOS)} puntos cargados.")

## 1. Prueba 1 — Tiempo actual en las 12 coordenadas

Para cada punto pedimos el bloque `current` con los campos que el panel realmente va a mostrar. Esto sirve a la vez como prueba funcional (¿devuelve datos coherentes?) y como medición de latencia y tamaño.

**Campos pedidos**: temperatura a 2 m, sensación térmica, humedad relativa, código de tiempo, indicador día/noche, velocidad y dirección del viento. Son los campos mínimos para alimentar la Vista 2 del panel.

In [ ]:
CAMPOS_CURRENT = "temperature_2m,relative_humidity_2m,apparent_temperature,is_day,weather_code,wind_speed_10m,wind_direction_10m"

def pedir_actual(lat, lon):
    params = {
        "latitude":  lat,
        "longitude": lon,
        "current":   CAMPOS_CURRENT,
        "timezone":  "auto",  # devuelve el timezone local del punto
    }
    t0 = time.time()
    r = requests.get(BASE, params=params, headers=HEADERS, timeout=15)
    elapsed_ms = (time.time() - t0) * 1000
    r.raise_for_status()
    return r.json(), elapsed_ms, len(r.content)

resultados_actual = []
latencias = []
tamanos = []

for nombre, lat, lon in PUNTOS:
    try:
        data, ms, size = pedir_actual(lat, lon)
        c = data["current"]
        resultados_actual.append({
            "punto":    nombre,
            "temp":     c["temperature_2m"],
            "sens":     c["apparent_temperature"],
            "hum":      c["relative_humidity_2m"],
            "code":     c["weather_code"],
            "is_day":   c["is_day"],
            "viento":   c["wind_speed_10m"],
            "vdir":     c["wind_direction_10m"],
            "tz":       data.get("timezone"),
            "elev":     data.get("elevation"),
            "ms":       ms,
            "size":     size,
        })
        latencias.append(ms)
        tamanos.append(size)
        print(f"OK  {nombre:35s}  {c['temperature_2m']:5.1f}°C  hum {c['relative_humidity_2m']:3d}%  code {c['weather_code']:2d}  ({ms:.0f} ms, {size} B)")
    except Exception as e:
        print(f"ERR {nombre:35s}  {e}")
    time.sleep(0.2)  # cortesía mínima, Open-Meteo no exige rate limit estricto pero no abusamos

print()
print(f"Latencia: media {mean(latencias):.0f} ms, mediana {median(latencias):.0f} ms, min {min(latencias):.0f} ms, max {max(latencias):.0f} ms")
print(f"Tamaño:   media {mean(tamanos):.0f} B,  mediana {median(tamanos):.0f} B,  min {min(tamanos)} B, max {max(tamanos)} B")

### Validación visual de resultados

Mostramos los 12 resultados en una tabla compacta. Buscamos:
- temperaturas plausibles para la fecha y el punto (sin valores absurdos como -50 o 80),
- humedad entre 0 y 100,
- timezone local correcto (las 12 deberían dar `Europe/Madrid` salvo Canarias que da `Atlantic/Canary`),
- elevación coherente con el punto (Albarracín ~1180 m, Garachico nivel del mar).

In [ ]:
print(f"{'Punto':35s} {'Temp':>6s} {'Sens':>6s} {'Hum':>5s} {'Code':>5s} {'Día':>4s} {'Viento':>8s} {'Elev':>7s} {'TZ':>20s}")
print("-" * 110)
for r in resultados_actual:
    dia = "sí" if r["is_day"] == 1 else "no"
    print(f"{r['punto']:35s} {r['temp']:5.1f}° {r['sens']:5.1f}° {r['hum']:4d}% {r['code']:5d} {dia:>4s} {r['viento']:6.1f}km/h {r['elev']:6.0f}m {r['tz']:>20s}")

## 2. Prueba 2 — Previsión por horas

Pedimos la previsión por horas para Madrid centro durante las próximas 24 horas. Validamos:
- el paso es horario,
- los campos hourly que pedimos vienen,
- los arrays están alineados (mismo número de elementos en `time` y en cada variable),
- el horizonte cubre las 12 h que necesita el panel.

**Campos**: temperatura, código de tiempo, probabilidad de precipitación, viento. Es lo mínimo para mostrar una franja horaria con icono y temperatura.

In [ ]:
CAMPOS_HOURLY = "temperature_2m,weather_code,precipitation_probability,wind_speed_10m"

params = {
    "latitude":  40.4168,   # Madrid centro
    "longitude": -3.7038,
    "hourly":    CAMPOS_HOURLY,
    "forecast_days": 1,
    "timezone":  "auto",
}
t0 = time.time()
r = requests.get(BASE, params=params, headers=HEADERS, timeout=15)
ms = (time.time() - t0) * 1000
r.raise_for_status()
data = r.json()
size = len(r.content)

h = data["hourly"]
print(f"Latencia: {ms:.0f} ms")
print(f"Tamaño:   {size} bytes")
print(f"Timezone respuesta: {data['timezone']} ({data['timezone_abbreviation']})")
print(f"Horas devueltas: {len(h['time'])}")
print(f"Variables alineadas: temp={len(h['temperature_2m'])}, code={len(h['weather_code'])}, prob={len(h['precipitation_probability'])}, viento={len(h['wind_speed_10m'])}")
print(f"Unidades: {data['hourly_units']}")
print()
print("Primeras 12 horas:")
print(f"{'Hora':>16s} {'Temp':>7s} {'Code':>5s} {'PrecP':>6s} {'Viento':>9s}")
for i in range(min(12, len(h["time"]))):
    print(f"{h['time'][i]:>16s} {h['temperature_2m'][i]:6.1f}° {h['weather_code'][i]:5d} {h['precipitation_probability'][i]:5d}% {h['wind_speed_10m'][i]:6.1f}km/h")

## 3. Prueba 3 — Mapeo de `weather_code` a categorías visuales

El panel necesita 6 categorías visuales para las animaciones de fondo de la Vista 2: **soleado, nublado, niebla, lluvia, nieve, tormenta**. Open-Meteo devuelve `weather_code` como entero según la tabla WMO 4677, con 28 valores posibles. Construimos la tabla de mapeo basada en la documentación oficial y verificamos que cubre los 28 códigos sin huecos.

**Tabla oficial WMO usada por Open-Meteo:**
- 0: Cielo despejado
- 1, 2, 3: Mayormente despejado, parcialmente nublado, cubierto
- 45, 48: Niebla y niebla con depósito
- 51, 53, 55: Llovizna ligera/moderada/densa
- 56, 57: Llovizna helada ligera/densa
- 61, 63, 65: Lluvia ligera/moderada/fuerte
- 66, 67: Lluvia helada ligera/fuerte
- 71, 73, 75: Nevada ligera/moderada/fuerte
- 77: Granos de nieve
- 80, 81, 82: Chubascos ligeros/moderados/violentos
- 85, 86: Chubascos de nieve ligeros/fuertes
- 95: Tormenta (ligera o moderada)
- 96, 99: Tormenta con granizo ligero/fuerte

In [ ]:
def categoria_visual(code):
    """
    Agrupa los 28 códigos WMO de Open-Meteo en 6 categorías visuales
    para alimentar las animaciones de fondo de la Vista 2 del panel.
    """
    if code == 0:
        return "soleado"
    if code in (1, 2, 3):
        return "nublado"
    if code in (45, 48):
        return "niebla"
    if code in (51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82):
        return "lluvia"
    if code in (71, 73, 75, 77, 85, 86):
        return "nieve"
    if code in (95, 96, 99):
        return "tormenta"
    return None  # Si aparece un código fuera de tabla, lo detectamos

# Verificación: los 28 códigos documentados por Open-Meteo deben caer en una categoría
codigos_oficiales = [
    0,
    1, 2, 3,
    45, 48,
    51, 53, 55,
    56, 57,
    61, 63, 65,
    66, 67,
    71, 73, 75, 77,
    80, 81, 82,
    85, 86,
    95,
    96, 99,
]

fallos = []
for c in codigos_oficiales:
    cat = categoria_visual(c)
    if cat is None:
        fallos.append(c)

print(f"Códigos verificados: {len(codigos_oficiales)}")
print(f"Fallos: {fallos if fallos else 'ninguno'}")
print()
print("Distribución por categoría:")
from collections import Counter
cuentas = Counter(categoria_visual(c) for c in codigos_oficiales)
for cat, n in sorted(cuentas.items()):
    print(f"  {cat:10s} → {n} códigos")

# Aplicar el mapeo a los códigos reales que han devuelto los 12 puntos
print()
print("Categorías observadas en las 12 coordenadas reales:")
for r in resultados_actual:
    print(f"  {r['punto']:35s}  code={r['code']:3d}  →  {categoria_visual(r['code'])}")

## 4. Prueba 4 — Comparación informal con AEMET

AEMET es la fuente oficial española. No queremos registrarnos para sacar una API key solo para una validación cruzada, pero sí queremos descartar que Open-Meteo se desvíe de forma seria en España. Para esta prueba, basta con anotar manualmente la temperatura que muestra AEMET en su web pública para una capital y compararla con el resultado de Open-Meteo en la misma coordenada.

**Procedimiento manual** (ejecutar en paralelo a esta celda):
1. Abrir https://www.aemet.es y buscar Madrid (o cualquier capital).
2. Anotar la temperatura observada y la condición que muestra AEMET.
3. Comparar con la celda siguiente.

Si la diferencia es menor a 1.5 °C consideramos a Open-Meteo coherente con AEMET para uso en el panel.

In [ ]:
# Reutilizamos el resultado de Madrid centro de la prueba 1 si está, si no lo pedimos.
data, ms, size = pedir_actual(40.4168, -3.7038)
c = data["current"]
print("Open-Meteo en Madrid centro AHORA:")
print(f"  Temperatura: {c['temperature_2m']} °C")
print(f"  Sensación:   {c['apparent_temperature']} °C")
print(f"  Humedad:     {c['relative_humidity_2m']} %")
print(f"  Código WMO:  {c['weather_code']} → {categoria_visual(c['weather_code'])}")
print(f"  Viento:      {c['wind_speed_10m']} km/h")
print(f"  Hora local:  {c['time']}")
print()
print("--> Compara este valor con el que muestra https://www.aemet.es para Madrid en este momento.")
print("--> Si la diferencia es < 1.5 °C, Open-Meteo se considera coherente con AEMET.")

## 5. Resumen y veredicto

Esta celda imprime el resumen que se trasladará a la ficha de API en `seguimiento_desarrollo_panel_viaje.docx`.

Criterios para considerar a Open-Meteo **apta**:
- 12/12 peticiones devuelven datos válidos.
- Temperaturas plausibles (rango razonable para España).
- Humedades 0-100 %.
- Mapeo de weather_code completo (28/28 códigos cubiertos).
- Latencia mediana < 500 ms.
- Tamaño de respuesta razonable (< 2 KB para current solo).
- Coherencia razonable con AEMET en el punto de control.

In [ ]:
print("=" * 60)
print("RESUMEN — Fase 1.3 Open-Meteo")
print("=" * 60)
print()
print(f"Puntos probados:           {len(resultados_actual)}/{len(PUNTOS)}")
print(f"Latencia mediana current:  {median(latencias):.0f} ms")
print(f"Latencia media current:    {mean(latencias):.0f} ms")
print(f"Tamaño mediano current:    {median(tamanos):.0f} bytes")
print(f"Códigos WMO mapeados:      {len(codigos_oficiales)}/28")
print()
temps = [r['temp'] for r in resultados_actual]
hums  = [r['hum']  for r in resultados_actual]
print(f"Rango de temperaturas observado: {min(temps):.1f}°C a {max(temps):.1f}°C")
print(f"Rango de humedades observado:    {min(hums)}% a {max(hums)}%")
print()
print("--> Rellenar la ficha de API en seguimiento_desarrollo_panel_viaje.docx")
print("    con estos valores tras la comparación visual con AEMET.")